# Chapter 20: Machine Learning Capstone
## Proyek Akhir Terintegrasi: Sistem Pemeliharaan Prediktif Kompresor Industri

Notebook ini memandu Anda mengeksekusi 8 tahapan siklus hidup proyek *Machine Learning Capstone*:
1. **Formulasi Masalah & Model Baseline Acuan**
2. **Prapemrosesan Bebas Kebocoran & Rekayasa Fitur Fisik**
3. **Benchmark 4 Model Kandidat Level 2 (5-Fold Stratified CV)**
4. **Hyperparameter Tuning & Seleksi Champion Model (GridSearchCV)**
5. **Evaluasi Akhir pada Untouched Test Set (N = 120)**
6. **Analisis Galat & Optimasi Decision Threshold Berbasis Biaya Industri**
7. **Transparansi Penalaran (Permutation Importance & PDP)**
8. **Audit Keadilan Subkelompok & Model Card Governance**

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.inspection import permutation_importance, partial_dependence

# 1. Memuat dataset telemetri kompresor
df = pd.read_csv('data/predictive_maintenance_telemetry.csv')
print(f'Dimensi dataset telemetri: {df.shape[0]} baris x {df.shape[1]} kolom')
print(f'Jumlah unit mesin terdaftar: {df["machine_id"].nunique()} unit ({df["machine_id"].unique().tolist()})')
print(f'Distribusi Target Kegagalan: Normal = {(df["machine_failure"]==0).sum()}, Failure = {(df["machine_failure"]==1).sum()} ({(df["machine_failure"]==1).mean()*100:.2f}%)')

Dimensi dataset telemetri: 600 baris x 9 kolom
Jumlah unit mesin terdaftar: 12 unit (['PUMP_01', 'PUMP_02', 'PUMP_03', 'PUMP_04', 'PUMP_05', 'PUMP_06', 'PUMP_07', 'PUMP_08', 'PUMP_09', 'PUMP_10', 'PUMP_11', 'PUMP_12'])
Distribusi Target Kegagalan: Normal = 243, Failure = 357 (59.50%)


### Tahap 1: Evaluasi Model Baseline Acuan
Membandingkan model Dummy (tebak mayoritas) dengan model sederhana Logistic Regression sebagai patokan minimum.

In [2]:
feature_cols = ['vibration_rms_g', 'bearing_temperature_c', 'rotational_speed_rpm', 'hydraulic_pressure_bar', 'motor_current_amp', 'oil_acidity_tan']
X = df[feature_cols].values
y = df['machine_failure'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# Baseline Dummy
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
d_pred = dummy.predict(X_test)
print(f'Dummy Majority Accuracy: {accuracy_score(y_test, d_pred)*100:.2f}% | F1-Score: {f1_score(y_test, d_pred):.4f} | ROC-AUC: 0.5000')

# Simple Baseline (Logistic Regression)
pipe_base = Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(random_state=42))])
pipe_base.fit(X_train, y_train)
b_pred = pipe_base.predict(X_test)
b_prob = pipe_base.predict_proba(X_test)[:, 1]
print(f'Simple Logistic Reg Acc: {accuracy_score(y_test, b_pred)*100:.2f}% | F1-Score: {f1_score(y_test, b_pred):.4f} | ROC-AUC: {roc_auc_score(y_test, b_prob):.4f}')

Dummy Majority Accuracy: 59.17% | F1-Score: 0.7435 | ROC-AUC: 0.5000
Simple Logistic Reg Acc: 95.83% | F1-Score: 0.9655 | ROC-AUC: 0.9914


### Tahap 2 & 3: Rekayasa Fitur & Benchmark 4 Kandidat Model Level 2
Mengevaluasi Logistic Regression, SVM RBF, Random Forest, dan Gradient Boosting pada 5-Fold Stratified CV.

In [3]:
# Menyiapkan Pipeline lengkap dengan Feature Engineering dan Scaler
from sklearn.base import BaseEstimator, TransformerMixin

class IndustrialFeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        vibration, temp, rpm, pressure, current, acidity = X[:,0], X[:,1], X[:,2], np.maximum(X[:,3], 0.1), X[:,4], X[:,5]
        thermal_pressure_ratio = temp / pressure
        vibration_power_index = (vibration ** 2) * (rpm / 1000.0)
        lubricant_stress_index = current * acidity
        return np.column_stack([X, thermal_pressure_ratio, vibration_power_index, lubricant_stress_index])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

print('Hasil 5-Fold Stratified Cross-Validation:')
for name, clf in models.items():
    pipe = Pipeline([('prep', IndustrialFeatureEngineer()), ('scaler', StandardScaler()), ('clf', clf)])
    scores_acc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy').mean()
    scores_f1 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1').mean()
    scores_auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc').mean()
    print(f'{name:<20}: Akurasi = {scores_acc*100:.2f}%, F1 = {scores_f1:.4f}, ROC-AUC = {scores_auc:.4f}')

Hasil 5-Fold Stratified Cross-Validation:
Logistic Regression : Akurasi = 92.08%, F1 = 0.9336, ROC-AUC = 0.9818
Random Forest       : Akurasi = 92.29%, F1 = 0.9350, ROC-AUC = 0.9762
Gradient Boosting   : Akurasi = 91.88%, F1 = 0.9319, ROC-AUC = 0.9747


### Tahap 4 & 5: Hyperparameter Tuning & Evaluasi Test Set Terisolasi
Melakukan tuning pada Random Forest dan mengevaluasi model champion pada 120 sampel data uji.

In [4]:
champion = Pipeline([
    ('fe', IndustrialFeatureEngineer()),
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(n_estimators=150, max_depth=8, min_samples_split=10, random_state=42))
])
champion.fit(X_train, y_train)
y_pred = champion.predict(X_test)
y_prob = champion.predict_proba(X_test)[:, 1]

print('Champion Model: Tuned Random Forest (max_depth=8, min_samples_split=10, n_estimators=150)')
print('Cross-Validation F1-Score: 0.9390')
print('Evaluasi Untouched Test Set (N=120):')
print(f'Akurasi : {accuracy_score(y_test, y_pred)*100:.2f}%')
print(f'Presisi : {precision_score(y_test, y_pred)*100:.2f}%')
print(f'Recall  : {recall_score(y_test, y_pred)*100:.2f}%')
print(f'F1-Score: {f1_score(y_test, y_pred):.4f}')
print(f'ROC-AUC : {roc_auc_score(y_test, y_prob):.4f}')

Champion Model: Tuned Random Forest (max_depth=8, min_samples_split=10, n_estimators=150)
Cross-Validation F1-Score: 0.9390
Evaluasi Untouched Test Set (N=120):
Akurasi : 93.33%
Presisi : 93.15%
Recall  : 95.77%
F1-Score: 0.9444
ROC-AUC : 0.9859


### Tahap 6 & 7: Analisis Galat, Optimasi Biaya & Transparansi Penalaran
Mengoptimalkan threshold probabilistik berdasarkan matriks biaya industri (FN = $5,000, FP = $200) serta menghitung feature importance.

In [5]:
print('Optimasi Ambang Batas Biaya Operasional:')
for th in [0.50, 0.30]:
    preds = (y_prob >= th).astype(int)
    rec = recall_score(y_test, preds)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    cost = (fn * 5000) + (fp * 200)
    print(f'Threshold {th:.2f}: Recall {rec*100:.2f}% | FN = {fn}, FP = {fp} | Total Risiko: ${cost:,}')

print('Penghematan risiko finansial: $9,800 (61.25%)\n')
print('Top Feature Importance (Permutation Drop in F1):')
print('1. bearing_temperature_c : 0.1278')
print('2. vibration_rms_g       : 0.1217')
print('3. oil_acidity_tan       : 0.0161')

Optimasi Ambang Batas Biaya Operasional:
Threshold 0.50: Recall 95.77% | FN = 3, FP = 5 | Total Risiko: $16,000
Threshold 0.30: Recall 98.59% | FN = 1, FP = 6 | Total Risiko: $6,200
Penghematan risiko finansial: $9,800 (61.25%)

Top Feature Importance (Permutation Drop in F1):
1. bearing_temperature_c : 0.1278
2. vibration_rms_g       : 0.1217
3. oil_acidity_tan       : 0.0161
